In [ ]:
import joblib
import pandas as pd
from loan_classifier.training import (
    build_model_registry, train_all_models,
    compare_models, evaluate_model,
    save_best_model, save_results_table
)

# Load data

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val   = pd.read_csv("../data/processed/X_val.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_val   = pd.read_csv("../data/processed/y_val.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()
preprocessor = joblib.load("../models/preprocessor.joblib")

# Build registry and train

In [ ]:
registry        = build_model_registry(preprocessor)
fitted_searches = train_all_models(registry, X_train, y_train)

# Compare on val set

In [ ]:
results = compare_models(fitted_searches, X_val, y_val)
display(results)
save_results_table(results, "../models/results_table.csv")

# Final evaluation on test (ONCE, after best model chosen)

In [ ]:
best_name = results.iloc[0]["model"]   # or pick manually from results
test_metrics = evaluate_model(fitted_searches[best_name], X_test, y_test, "test")
print(test_metrics)

# Visualization

In [ ]:
from loan_classifier.training import classification_report_markdown
from loan_classifier.training import plot_roc_curve
from loan_classifier.training import plot_confusion_matrix

In [ ]:
best_name = results.iloc[0]["model"]
search = fitted_searches[best_name]

plot_roc_curve(search, X_val, y_val, title=f"{best_name} ROC (val)")

In [ ]:
plot_confusion_matrix(search, X_val, y_val, title=f"{best_name} Confusion Matrix (val)")

In [ ]:
md = classification_report_markdown(search, X_val, y_val)
print(md)  # for quick view

In [ ]:
# In Jupyter you can also display as Markdown:
from IPython.display import Markdown, display
display(Markdown("### Classification report (val)\n\n" + md))

# Save best model

In [ ]:
save_best_model(fitted_searches, best_name, "../models/best_model.joblib")

# Extending later

In [ ]:
# from sklearn.ensemble import ExtraTreesClassifier
# registry.register(
#     name="ExtraTrees",
#     estimator=ExtraTreesClassifier(random_state=42),
#     param_grid={"classifier__n_estimators": [100, 200]},
#     preprocessor=preprocessor
# )
# fitted_searches.update(train_all_models(registry, X_train, y_train))